# Sanity checks: published and independent Yamada references

This is the canonical correctness notebook for the Yamada implementation. It checks published closed forms, structural identities, an explicit published $K_4$ value, an independent Negami edge-subset implementation, agreement of the public backends, and crossing/mirror behavior.

All comparisons to explicit literature Laurent polynomials use `normalize=False`. Every section raises `AssertionError` on disagreement.


## References

The checks use the conventions/results of:

1. S. Yamada, *An invariant of spatial graphs*, Journal of Graph Theory **13** (1989), 537–551.
2. M. Li, F. Lei, F. Li, A. Vesnin, *On Yamada polynomial of spatial graphs obtained by edge replacements* (2018), arXiv:1801.09075.
3. S. R. T. Peddada et al., *Enumeration and Identification of Unique 3D Spatial Topologies of Interconnected Engineering Systems Using Spatial Graphs*, arXiv:2107.13724.
4. A. A. Dobrynin and A. Vesnin, *The Yamada polynomial for graphs embedded knot-wise into three-dimensional space* (1996).

Backend agreement is tested, but is not used as a substitute for the external literature checks.


In [ ]:
from pathlib import Path
import sys
import networkx as nx
import numpy as np
import sympy as sp

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from knotted_graph.invariants.yamada import (
    compute_negami,
    compute_negami_recursive,
    compute_yamada_polynomial_recursive,
)
from knotted_graph.projection import compute_yamada_polynomial

A = sp.Symbol("A")
x, y = sp.symbols("x y")
sigma = A + 1 + A**-1

def same_polynomial(left, right):
    return sp.simplify(sp.together(sp.expand(left - right))) == 0

def require_same(label, computed, expected):
    if not same_polynomial(computed, expected):
        raise AssertionError(
            f"{label} failed.\ncomputed={sp.expand(computed)}\nexpected={sp.expand(expected)}"
        )
    print(f"PASS  {label}")


In [ ]:
def tree_graph(q):
    return nx.MultiGraph(nx.path_graph(q + 1))

def cycle_graph(n):
    if n == 1:
        G = nx.MultiGraph()
        G.add_node(0)
        G.add_edge(0, 0)
        return G
    if n == 2:
        G = nx.MultiGraph()
        G.add_nodes_from([0, 1])
        G.add_edge(0, 1)
        G.add_edge(0, 1)
        return G
    return nx.MultiGraph(nx.cycle_graph(n))

def bouquet_graph(q):
    G = nx.MultiGraph()
    G.add_node(0)
    for _ in range(q):
        G.add_edge(0, 0)
    return G

def theta_graph(s):
    G = nx.MultiGraph()
    G.add_nodes_from([0, 1])
    for _ in range(s):
        G.add_edge(0, 1)
    return G

def one_point_union(G1, G2):
    G1 = nx.convert_node_labels_to_integers(G1, first_label=0)
    G2 = nx.convert_node_labels_to_integers(G2, first_label=G1.number_of_nodes() - 1)
    return nx.MultiGraph(nx.compose(G1, G2))


## 1. Published closed-form families

With $\sigma=A+1+A^{-1}$, Li–Lei–Li–Vesnin give

\[
H(T_q)=0,\qquad H(C_n)=\sigma,\qquad
H(B_q)=(-1)^{q-1}\sigma^q,
\]

and

\[
H(\Theta_s)=\frac{\sigma+(-\sigma)^s}{\sigma+1}.
\]


In [ ]:
for q in range(1, 7):
    require_same(f"Tree T_{q}", compute_yamada_polynomial_recursive(tree_graph(q), A), 0)

for n in range(1, 8):
    require_same(f"Cycle C_{n}", compute_yamada_polynomial_recursive(cycle_graph(n), A), sigma)

for q in range(1, 7):
    require_same(
        f"Bouquet B_{q}",
        compute_yamada_polynomial_recursive(bouquet_graph(q), A),
        (-1)**(q - 1) * sigma**q,
    )

for s in range(1, 9):
    require_same(
        f"Theta_{s}",
        compute_yamada_polynomial_recursive(theta_graph(s), A),
        (sigma + (-sigma)**s) / (sigma + 1),
    )


## 2. Bridge and one-point-union identities

A graph containing an isthmus has $H(G)=0$, while $H(G_1\cdot G_2)=-H(G_1)H(G_2)$ for a one-point union.


In [ ]:
left = cycle_graph(3)
right = nx.relabel_nodes(cycle_graph(4), lambda node: node + 10)
bridged = nx.compose(left, right)
bridged.add_edge(0, 10)
require_same(
    "Composite graph containing an isthmus",
    compute_yamada_polynomial_recursive(bridged, A),
    0,
)

G1 = theta_graph(3)
G2 = bouquet_graph(2)
wedge = one_point_union(G1, G2)
require_same(
    "One-point union",
    compute_yamada_polynomial_recursive(wedge, A),
    -compute_yamada_polynomial_recursive(G1, A)
    * compute_yamada_polynomial_recursive(G2, A),
)


## 3. Published planar $K_4$

Dobrynin and Vesnin tabulate

\[
H(K_4)=A^3+2A+2A^{-1}+A^{-3}.
\]


In [ ]:
K4 = nx.MultiGraph(nx.complete_graph(4))
require_same(
    "Planar K4",
    compute_yamada_polynomial_recursive(K4, A),
    A**3 + 2*A + 2*A**-1 + A**-3,
)


## 4. Independent Negami definition

For small graphs the direct edge-subset definition of the two-variable Negami polynomial is compared to the deletion–contraction implementation before specialization.


In [ ]:
small_graphs = {
    "Bouquet B2": bouquet_graph(2),
    "Cycle C3": cycle_graph(3),
    "Theta3": theta_graph(3),
    "K4": K4,
    "Tree T2": tree_graph(2),
}
for name, G in small_graphs.items():
    subset_h = compute_negami(G, x, y)
    recursive_h = compute_negami_recursive(G, x, y)
    require_same(f"{name}: direct vs recursive Negami", subset_h, recursive_h)
    specialized = recursive_h.xreplace(
        {x: sp.Integer(-1), y: -A - 2 - A**-1}
    )
    require_same(
        f"{name}: Negami specialization",
        specialized,
        compute_yamada_polynomial_recursive(G, A),
    )


## 5. Public spatial-graph API on a planar theta graph

Peddada et al. give $R(\theta)=B-B^2$ with $B=A+1+A^{-1}$. The public `negami` and `recursive` backends must both reproduce it.


In [ ]:
def embedded_planar_theta():
    G = nx.MultiGraph()
    G.add_node("u", pos=np.array([-2.0, 0.0, 0.0]))
    G.add_node("v", pos=np.array([2.0, 0.0, 0.0]))
    curves = [
        np.array([[-2,0,0],[-1,1,0],[1,1,0],[2,0,0]], dtype=float),
        np.array([[-2,0,0],[-1,0,0],[1,0,0],[2,0,0]], dtype=float),
        np.array([[-2,0,0],[-1,-1,0],[1,-1,0],[2,0,0]], dtype=float),
    ]
    for pts in curves:
        G.add_edge("u", "v", pts=pts)
    return G

planar_theta = embedded_planar_theta()
target = sigma - sigma**2
results = {}
for method in ("negami", "recursive"):
    result = compute_yamada_polynomial(
        planar_theta, A,
        rotation_angles=(0.0, 0.0, 0.0),
        normalize=False, n_jobs=1, method=method, return_result=True,
    )
    if result.projection.num_crossings != 0:
        raise AssertionError("Planar theta unexpectedly has a crossing.")
    require_same(f"Public {method} backend, planar theta", result.polynomial, target)
    results[method] = result.polynomial
require_same("Public backend agreement, planar theta", results["negami"], results["recursive"])


## 6. One-crossing theta and its mirror

The one-crossing representative must differ from the planar theta by one of the Laurent units $-A$ or $-A^{-1}$; mirroring exchanges the two.


In [ ]:
def embedded_one_crossing_theta(*, mirror=False):
    zsign = -1.0 if mirror else 1.0
    G = nx.MultiGraph()
    G.add_node("u", pos=np.array([-2.0, 0.0, 0.0]))
    G.add_node("v", pos=np.array([2.0, 0.0, 0.0]))
    curves = [
        np.array([[-2,0,0],[-1,-1,0.5*zsign],[1,1,0.5*zsign],[2,0,0]], dtype=float),
        np.array([[-2,0,0],[-1,1,-0.5*zsign],[1,-1,-0.5*zsign],[2,0,0]], dtype=float),
        np.array([[-2,0,0],[-1,2,0],[1,2,0],[2,0,0]], dtype=float),
    ]
    for pts in curves:
        G.add_edge("u", "v", pts=pts)
    return G

def public_yamada(G, method):
    return compute_yamada_polynomial(
        G, A, rotation_angles=(0.0,0.0,0.0),
        normalize=False, n_jobs=1, method=method, return_result=True,
    )

crossed = {m: public_yamada(embedded_one_crossing_theta(), m) for m in ("negami","recursive")}
mirrored = {m: public_yamada(embedded_one_crossing_theta(mirror=True), m) for m in ("negami","recursive")}

for label, group in (("crossed", crossed), ("mirror", mirrored)):
    for method, result in group.items():
        if result.projection.num_crossings != 1:
            raise AssertionError(f"{label}/{method}: expected one crossing.")
    require_same(f"{label}: backend agreement", group["negami"].polynomial, group["recursive"].polynomial)

crossed_factor = sp.simplify(crossed["recursive"].polynomial / target)
mirror_factor = sp.simplify(mirrored["recursive"].polynomial / target)
if crossed_factor not in {-A, -A**-1}:
    raise AssertionError(f"Unexpected crossed theta factor {crossed_factor}")
if mirror_factor not in {-A, -A**-1}:
    raise AssertionError(f"Unexpected mirror theta factor {mirror_factor}")
if sp.simplify(crossed_factor * mirror_factor - 1) != 0:
    raise AssertionError("Mirroring did not exchange -A and -A^-1.")
require_same(
    "Mirror relation R_mirror(A)=R(A^-1)",
    mirrored["recursive"].polynomial,
    sp.expand(crossed["recursive"].polynomial.subs(A, A**-1, simultaneous=True)),
)


## Acceptance criterion

A successful execution means every external-reference, structural, independent-implementation, backend, crossing, and mirror assertion above passed. This notebook contains no speed claim; it is the correctness gate for the performance notebooks that follow.
